# Week 12: Generative AI, Transformers & Diffusion
## Notebook 10: Score-Based Generative Modeling & Stochastic Differential Equations

### 1. Conceptual Intuition & Theoretical Background
Diffusion models can be unified under the framework of **Stochastic Differential Equations (SDEs, Song et al., 2020)**.

**1. Forward SDE (Noising Process):**
Diffuses clean data $x_0 \sim p_0$ into unstructured Gaussian noise over continuous time $t \in [0, T]$:
$$dx = f(x, t) dt + g(t) dw$$
Where $f(x, t)$ is the drift coefficient, $g(t)$ is the diffusion coefficient, and $w$ is a standard Brownian motion.

**2. Reverse-Time SDE (Generation Process):**
Anderson's Theorem (1982) proves that the reverse trajectory satisfies:
$$dx = \left[ f(x, t) - g(t)^2 \nabla_x \ln p_t(x) \right] dt + g(t) d\bar{w}$$
Where $\nabla_x \ln p_t(x)$ is the **Stein Score Function** (the gradient of the log-probability density).

A neural network $s_\theta(x, t)$ learns to approximate this score function via **Denoising Score Matching**:
$$\mathcal{L}_{\text{DSM}}(\theta) = \mathbb{E}_{t, x_0, x_t}\left[ \left\| s_\theta(x_t, t) - \nabla_{x_t} \ln p(x_t \mid x_0) \right\|^2 \right]$$

---

### 2. Scratch Python Implementation: Langevin Dynamics Score Ascent

In [ ]:
import numpy as np

# Ground Truth Target Density: Bimodal Gaussian Mixture in 1D
# p(x) = 0.5 * N(-3, 0.5^2) + 0.5 * N(3, 0.5^2)
def true_score_function(x):
    """
    Computes analytical Stein Score: grad_x log p(x)
    """
    w1, mu1, sig1 = 0.5, -3.0, 0.5
    w2, mu2, sig2 = 0.5, 3.0, 0.5
    
    p1 = w1 * np.exp(-0.5 * ((x - mu1) / sig1)**2) / (sig1 * np.sqrt(2 * np.pi))
    p2 = w2 * np.exp(-0.5 * ((x - mu2) / sig2)**2) / (sig2 * np.sqrt(2 * np.pi))
    p_total = p1 + p2 + 1e-9
    
    # Derivative d/dx p(x)
    dp1 = p1 * (-(x - mu1) / (sig1 ** 2))
    dp2 = p2 * (-(x - mu2) / (sig2 ** 2))
    
    score = (dp1 + dp2) / p_total
    return score

# Annealed Langevin Dynamics Sampling from pure random initial states
def annealed_langevin_dynamics(n_samples=500, n_steps=200, step_size=0.05):
    # Initialize from uniform distribution over [-8, 8]
    x = np.random.uniform(-8.0, 8.0, n_samples)
    
    for step in range(n_steps):
        # Score step + Brownian noise injection
        score = true_score_function(x)
        noise = np.random.normal(0, 1.0, n_samples)
        x = x + 0.5 * step_size * score + np.sqrt(step_size) * noise
        
    return x

np.random.seed(42)
sampled_particles = annealed_langevin_dynamics()

# Verify bimodal mode clustering around -3.0 and +3.0
mode_neg = np.sum(sampled_particles < 0)
mode_pos = np.sum(sampled_particles >= 0)

print("--- Score-Based Generative Langevin Sampling ---")
print(f"Total Generated Particles:         {len(sampled_particles)}")
print(f"Particles in Negative Mode (-3):   {mode_neg} ({mode_neg/len(sampled_particles)*100:.1f}%)")
print(f"Particles in Positive Mode (+3):   {mode_pos} ({mode_pos/len(sampled_particles)*100:.1f}%)")
print(f"Negative Mode Empirical Mean:      {np.mean(sampled_particles[sampled_particles < 0]):.2f} (Target: -3.00)")
print(f"Positive Mode Empirical Mean:      {np.mean(sampled_particles[sampled_particles >= 0]):.2f} (Target: +3.00)")

### 3. Step-by-Step Code Walkthrough
1. **Stein Score Formulation:** Evaluates the vector field $\nabla_x \ln p(x)$, which points toward high-density modes.
2. **Langevin Dynamics:** Balances deterministic score ascent with stochastic noise injection.
3. **Mode Equilibrium:** Generates particles matching the target bimodal distribution without computing the normalizing partition function.

---

### 4. Industry Applications & Trade-offs
- **High-Fidelity Audio & Molecule Generation:** Generating valid 3D molecular conformations using continuous-time reverse SDEs.
- **Trade-off:** Score estimation in low-density manifolds is noisy without multi-scale noise annealing.

---

### 5. Top Interview Questions & Answers
**Q1: Why does score-based modeling eliminate the need to compute the normalizing partition function $Z$?**
> **Answer:** The score is the gradient of the log probability density: $\nabla_x \ln p(x) = \nabla_x \ln \frac{\tilde{p}(x)}{Z} = \nabla_x \ln \tilde{p}(x) - \nabla_x \ln Z = \nabla_x \ln \tilde{p}(x)$. Because $Z$ is independent of $x$, its gradient is zero, bypassing the intractable partition integral.

**Q2: What is the connection between the Probability Flow ODE and the Reverse-Time SDE?**
> **Answer:** Every reverse-time SDE has an associated deterministic Probability Flow ODE that shares the exact same marginal probability densities $p_t(x)$ at all time points $t$, enabling fast deterministic sampling and exact likelihood computation.